# Week 37, Day 4: Logging

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w037d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import logging

logging.debug("checking the basket")
logging.info("basket has 3 items")
logging.warning("item 2 is nearly out of stock")
logging.error("payment service didn't answer")
print("order saved")

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

def load(rows):
    logging.debug("raw rows: %r", rows)
    logging.info("loading %d rows", len(rows))
    good = [r for r in rows if r.strip()]
    if len(good) < len(rows):
        logging.warning("skipped %d blank rows", len(rows) - len(good))
    return good

print(load(["a", "", "b", "  "]))

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
shop = logging.getLogger("shop")
storage = logging.getLogger("shop.storage")
payments = logging.getLogger("shop.payments")

storage.setLevel(logging.DEBUG)          # detail for storage only

storage.debug("opened orders.json")
payments.debug("contacting the bank")     # hidden: payments uses shop's level, which is root's WARNING
payments.warning("the bank is slow")
print(storage.parent.name, payments.getEffectiveLevel(), storage.getEffectiveLevel())

In [ ]:
import io
import logging
import sys

log = logging.getLogger("importer")
log.setLevel(logging.DEBUG)
log.propagate = False                  # don't also pass messages up to the root's handlers

screen = logging.StreamHandler(sys.stdout)
screen.setLevel(logging.INFO)
screen.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

problems = io.StringIO()               # stands in for a file of problems
saved = logging.StreamHandler(problems)
saved.setLevel(logging.WARNING)
saved.setFormatter(logging.Formatter("%(name)s|%(levelname)s|%(message)s"))

log.addHandler(screen)
log.addHandler(saved)

log.debug("row 1 looks fine")         # nowhere: both handlers want INFO or above
log.info("imported 40 rows")          # screen only
log.warning("row 7 had no email")     # both
print("--- saved problems ---")
print(problems.getvalue(), end="")

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", stream=sys.stdout)
log = logging.getLogger("orders")

def save_all(orders):
    saved = 0
    for order in orders:
        try:
            total = order["qty"] * order["price"]
        except KeyError:
            log.exception("order %s is missing a field", order.get("id", "?"))
            continue
        saved += 1
        log.info("order %s: %d", order["id"], total)
    return saved

print(save_all([{"id": 1, "qty": 2, "price": 300}, {"id": 2, "qty": 1}]))

> **Key idea.** `print` is for the program's output; `logging` for its story. Messages have levels (`DEBUG` 10, `INFO` 20, `WARNING` 30, `ERROR` 40, `CRITICAL` 50), and only those at or above the level in force are shown; the default is `WARNING`. Call `basicConfig` once, first, and pass values as arguments (`log.info("%d rows", n)`). `getLogger(__name__)` gives each module a logger in a dotted tree; levels are inherited and messages propagate up. Handlers choose where messages go and formatters how they look, each handler with its own level. `log.exception` adds the traceback; log an error once, where it's handled, and never log secrets. Libraries only log; the application configures.

### Exercise: A logger of your own

Write `make_logger(name, stream, level)`, which returns the logger called `name`, set to `level`, not propagating, with exactly **one** handler: a `StreamHandler` writing to `stream`, with the format `"%(levelname)s [%(name)s] %(message)s"`. Calling it twice with the same name mustn't add a second handler (remove any existing ones first).

In [ ]:
import logging

def make_logger(name, stream, level):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\nimport logging\ndef test_logger():\n    \"the logger\"\n    s = io.StringIO()\n    log = make_logger(\"ex.setup\", s, logging.INFO)\n    log.debug(\"hidden\")\n    log.info(\"shown %d\", 1)\n    log.error(\"bad\")\n    assert s.getvalue() == \"INFO [ex.setup] shown 1\\nERROR [ex.setup] bad\\n\", repr(s.getvalue())\n    assert log.propagate is False\ndef test_twice():\n    \"no duplicates\"\n    s1, s2 = io.StringIO(), io.StringIO()\n    make_logger(\"ex.twice\", s1, logging.DEBUG)\n    log = make_logger(\"ex.twice\", s2, logging.DEBUG)\n    log.debug(\"once\")\n    assert len(log.handlers) == 1 and s1.getvalue() == \"\" and s2.getvalue() == \"DEBUG [ex.twice] once\\n\"")

### Exercise: The right level

Write `check_readings(readings, log)`, which goes through a list of temperature readings (numbers, or `None` for a missing one) and logs each with the right level, then returns the average of the real readings (or `None` if there are none):

- each reading: `DEBUG` `"reading N: VALUE"` (N from 1)
- a `None`: `WARNING` `"reading N is missing"`
- a reading above 40: `ERROR` `"reading N is too hot: VALUE"`
- at the end: `INFO` `"average of K readings: AVG"`, with the average to 1 decimal place (skip this line if there are none)

Pass values as arguments to the logging calls, not with f-strings.

In [ ]:
def check_readings(readings, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\nimport logging\ndef capture(level=logging.DEBUG):\n    s = io.StringIO()\n    log = logging.getLogger(\"ex.levels\")\n    log.handlers.clear()\n    h = logging.StreamHandler(s)\n    h.setFormatter(logging.Formatter(\"%(levelname)s %(message)s\"))\n    log.addHandler(h)\n    log.setLevel(level)\n    log.propagate = False\n    return log, s\ndef test_levels():\n    \"levels\"\n    log, s = capture()\n    assert check_readings([20, None, 41.5, 18], log) == 26.5\n    assert s.getvalue().splitlines() == [\"DEBUG reading 1: 20\", \"DEBUG reading 2: None\", \"WARNING reading 2 is missing\", \"DEBUG reading 3: 41.5\", \"ERROR reading 3 is too hot: 41.5\", \"DEBUG reading 4: 18\", \"INFO average of 3 readings: 26.5\"], s.getvalue()\ndef test_quiet():\n    \"at WARNING\"\n    log, s = capture(logging.WARNING)\n    assert check_readings([None, None], log) is None\n    assert s.getvalue().splitlines() == [\"WARNING reading 1 is missing\", \"WARNING reading 2 is missing\"]\n    import ast\n    assert not any(isinstance(n, ast.JoinedStr) for n in ast.walk(ast.parse(source))), \"pass values as arguments, not with f-strings\"")

### Exercise: Turning up one branch

Write `configure()`, which sets the logger `"shop"` to `INFO`, `"shop.storage"` to `DEBUG` and `"shop.payments"` to `ERROR`, and returns nothing. The checks then look at the levels in force for loggers further down the tree.

In [ ]:
import logging

def configure():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import logging\ndef test_tree():\n    \"effective levels\"\n    configure()\n    get = lambda name: logging.getLogger(name).getEffectiveLevel()\n    assert get(\"shop\") == logging.INFO and get(\"shop.basket\") == logging.INFO\n    assert get(\"shop.storage\") == logging.DEBUG and get(\"shop.storage.json\") == logging.DEBUG\n    assert get(\"shop.payments.cards\") == logging.ERROR")

### Exercise: Logging what failed

Write `divide_all(pairs, log)`, which returns a list of `a / b` for each `(a, b)` pair, with `None` where the division fails with a `ZeroDivisionError` or `TypeError`. For each failure, call `log.exception` with the message `"couldn't divide A by B"` (values as arguments, shown with `%r`), so the traceback is recorded. Successful divisions aren't logged.

In [ ]:
def divide_all(pairs, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\nimport logging\ndef test_divide():\n    \"with tracebacks\"\n    s = io.StringIO()\n    log = logging.getLogger(\"ex.errs\")\n    log.handlers.clear()\n    log.addHandler(logging.StreamHandler(s))\n    log.propagate = False\n    assert divide_all([(6, 3), (1, 0), (\"x\", 2), (5, 2)], log) == [2.0, None, None, 2.5]\n    text = s.getvalue()\n    assert \"couldn't divide 1 by 0\" in text and \"couldn't divide 'x' by 2\" in text, text\n    assert text.count(\"Traceback\") == 2 and \"ZeroDivisionError\" in text and \"TypeError\" in text, \"use log.exception, so the tracebacks are recorded\"")

### Bug hunt: Configured too late

This program should show its debug message, but the first `logging.warning` call quietly sets up logging with the default level before `basicConfig` runs, and then `basicConfig` does nothing. Move the configuration so the output is `WARNING:root:starting up` then `DEBUG:root:reading settings`.

In [ ]:
import logging

logging.warning("starting up")
logging.basicConfig(level=logging.DEBUG)
logging.debug("reading settings")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"configure first\"\n    expect_output(\"WARNING:root:starting up\\nDEBUG:root:reading settings\")")

### Bug hunt: Every message twice

Each message appears twice: once from the `"app"` logger's own handler, and again when it propagates up to the root's handler from `basicConfig`. Stop it propagating, so the output is `app: ready` then `app: saved 3 rows`.

In [ ]:
import logging
import sys

logging.basicConfig(format="%(name)s: %(message)s", stream=sys.stdout)
log = logging.getLogger("app")
log.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(name)s: %(message)s"))
log.addHandler(handler)

log.info("ready")
log.info("saved %d rows", 3)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"once each\"\n    expect_output(\"app: ready\\napp: saved 3 rows\")")

### Bug hunt: The handler was ready, the logger wasn't

The handler is set to `DEBUG`, but the logger itself has no level, so it uses the root's `WARNING` and the info message never reaches the handler. Fix it so the output is `INFO cache warmed` then `WARNING cache nearly full`.

In [ ]:
import logging
import sys

log = logging.getLogger("cache")
log.propagate = False
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
log.addHandler(handler)

log.info("cache warmed")
log.warning("cache nearly full")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both gates\"\n    expect_output(\"INFO cache warmed\\nWARNING cache nearly full\")")

### Puzzle: Setting up a logger

Arrange the lines into a function that gives a logger one handler writing to a stream, with a format. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import logging

top = logging.getLogger("t")
mid = logging.getLogger("t.mid")
low = logging.getLogger("t.mid.low")
top.setLevel(logging.ERROR)
mid.setLevel(logging.DEBUG)

a = logging.INFO
b = low.getEffectiveLevel()
c = logging.getLogger("t.other").isEnabledFor(logging.WARNING)
d = low.parent.name
e = logging.getLogger("t.mid") is mid

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # d
t = ___   # e

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 20 and q == 10 and r is False, \"p, q and r\"\n    assert s == \"t.mid\" and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a handler that remembers

Handlers are classes: subclass `logging.Handler` and write `emit(self, record)`. Build:

- `MemoryHandler`, a handler that keeps every record it receives in a list `self.records` (call `super().__init__(level)` in `__init__(self, level=logging.NOTSET)`), and has `counts()`, a dictionary from level name to how many records had it, and `messages(level)`, the finished messages (`record.getMessage()`) of records at **exactly** that level name, in order.
- `import_people(rows, log)`, which turns rows like `"Ada,36"` into `(name, age)` tuples and returns them. It logs `INFO` `"importing %d rows"` first; for a row without a comma or with a bad age it logs `WARNING` `"skipping row %d: %r"` (row numbers from 1) and skips it; and at the end, if any rows were skipped, `ERROR` `"%d of %d rows skipped"`.

In [ ]:
import logging

class MemoryHandler(logging.Handler):
    ___

def import_people(rows, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import logging\ndef fresh(level=logging.DEBUG):\n    log = logging.getLogger(\"ex.challenge\")\n    log.handlers.clear()\n    log.setLevel(logging.DEBUG)\n    log.propagate = False\n    h = MemoryHandler(level)\n    log.addHandler(h)\n    return log, h\ndef test_handler():\n    \"the handler\"\n    log, h = fresh()\n    log.info(\"a %d\", 1)\n    log.warning(\"b\")\n    log.info(\"c\")\n    assert [r.getMessage() for r in h.records] == [\"a 1\", \"b\", \"c\"]\n    assert h.counts() == {\"INFO\": 2, \"WARNING\": 1} and h.messages(\"INFO\") == [\"a 1\", \"c\"] and h.messages(\"ERROR\") == []\n    log, h = fresh(logging.WARNING)\n    log.info(\"hidden\")\n    log.error(\"kept\")\n    assert h.messages(\"ERROR\") == [\"kept\"] and h.counts() == {\"ERROR\": 1}, \"the handler's level should be respected\"\ndef test_import():\n    \"importing\"\n    log, h = fresh()\n    got = import_people([\"Ada,36\", \"Bo\", \"Chi,old\", \"Dee,40\"], log)\n    assert got == [(\"Ada\", 36), (\"Dee\", 40)]\n    assert h.messages(\"INFO\") == [\"importing 4 rows\"]\n    assert h.messages(\"WARNING\") == [\"skipping row 2: 'Bo'\", \"skipping row 3: 'Chi,old'\"]\n    assert h.messages(\"ERROR\") == [\"2 of 4 rows skipped\"]\n    log, h = fresh()\n    assert import_people([\"Ed,5\"], log) == [(\"Ed\", 5)] and h.counts() == {\"INFO\": 1}")

## Warm-up

Only two of the four messages appeared. Out of the box, `logging` shows **warnings and worse**, and hides the everyday detail. That's the first thing `print` can't do: decide, without editing the code, which messages matter right now.

## Why not print?

`print` writes the program's **output**: the answer, the report, the table the user asked for. Logging records the program's **story**: what it did, what it noticed, what went wrong. Mixing them causes problems:

| Need | With `print` | With `logging` |
|---|---|---|
| Hide the detail normally, show it when debugging | delete or comment out lines | change one level setting |
| Know how serious each message is | write "WARNING" by hand | every message has a level |
| Send errors to a file, everything else to the screen | write your own code | add handlers |
| Know which part of the program said it | write the name by hand | every logger has a name |
| Keep the program's output clean for a pipe (Week 21) | impossible | log messages go to stderr by default |

So: use `print` (or `sys.stdout`) for what the user asked for, and `logging` for everything the program wants to say about itself. A tiny script can happily use `print`; anything that runs for a long time, or that other people will run, should log.

## Levels and basicConfig

Every message has one of five standard levels, each a number:

| Level | Number | For |
|---|---|---|
| `DEBUG` | 10 | detail for finding problems: values, decisions, steps |
| `INFO` | 20 | normal events worth recording: started, loaded 300 rows, saved |
| `WARNING` | 30 | something unexpected that the program coped with: a missing setting, using a default |
| `ERROR` | 40 | something failed: this order couldn't be saved |
| `CRITICAL` | 50 | the program can't carry on |

A logger shows messages **at or above** its level. `logging.basicConfig` sets up the simplest arrangement in one call, and must come **before** any logging:

- `level=logging.INFO` shows `INFO` and above, so the `DEBUG` line stays hidden. Set `level=logging.DEBUG` and it appears, with no other change.
- `format` chooses what each line looks like; `%(levelname)-8s` pads the level to 8 characters. Real programs usually add `%(asctime)s` for the time; this lesson leaves it out so the outputs stay the same every run.
- Pass values as **extra arguments** with `%` placeholders, not as an f-string: `logging.info("loading %d rows", n)`. The text is only built if the message will actually be shown, which matters for debug messages in a loop, and tools that collect logs can group messages by their unformatted text.

## Named loggers

The functions `logging.info(...)` and so on use the **root** logger. Real programs give each module its own logger, named after the module:

```python
import logging

log = logging.getLogger(__name__)      # in shop/storage.py, the name is "shop.storage"
```

Names with dots form a **tree**: `shop.storage` is a child of `shop`, which is a child of the root. A logger with no level of its own uses its parent's, and messages travel **up** the tree to the parents' handlers (this is called propagation). So you can turn on detail for one part of a program:

- `logging.getLogger(name)` returns the **same** logger every time for the same name, so every module can ask for it without passing it around.
- `getEffectiveLevel()` gives the level actually in force, found by walking up the tree until some logger has one set.
- Create the parent (`shop`) before children if you want `.parent` to show it: a logger's parent is the nearest **existing** one.

In the browser, every program runs as `__main__`, so `getLogger(__name__)` gives a logger called `__main__`. In a package (Week 20), it gives `shop.storage` and so on, for free.

## Handlers and formatters

A logger decides **whether** a message is worth recording. **Handlers** decide **where** it goes, and **formatters** decide **what it looks like**. One logger can have several handlers, each with its own level:

A message passes **two** gates: the logger's level, then each handler's level. The logger's level is the main switch; handlers' levels split the messages up.

- `logging.StreamHandler(stream)` writes to any file-like object (Week 30's duck typing): `sys.stderr` (the default), `sys.stdout`, or an `io.StringIO`, which is handy for tests.
- `logging.FileHandler("app.log")` appends to a file.
- Common format fields: `%(levelname)s`, `%(name)s`, `%(message)s`, `%(asctime)s`, `%(funcName)s`, `%(lineno)d`.

Set `propagate = False` when a logger has its own handlers and you don't want each message repeated by the root's handlers too. Forgetting it is the usual cause of every message appearing twice.

## Logging errors

Inside an `except` block, `log.exception(...)` logs at `ERROR` level **and** adds the traceback, which is usually the most useful thing in a log:

Where should an error be logged? **Once, where it's handled.** If every layer logs an error and re-raises it, one problem fills the log with the same traceback five times. A good pattern:

- Low layers **raise** (with `from`, Day 2) and don't log errors they pass up.
- The place that **handles** the error (skips the row, shows a message, returns an exit code) logs it, with `exception` if the traceback helps.
- `warning` is for problems the program coped with; `error` for things that failed.

And never log secrets or private details: passwords, keys, card numbers, whole personal records. Logs are copied, kept and read by many people. Log an id, not the record.

**Libraries and applications.** Code meant to be imported (a library, or your own packages) only does `log = logging.getLogger(__name__)` and logs. It never calls `basicConfig` or adds handlers, because it doesn't know where its users want messages to go. The **application**, in its composition root (Week 36), configures logging once, at start-up.

## Practice

## Quiz

## Daily challenge

## Before you go

Take a program of yours that prints progress messages and change them to log calls at sensible levels, with the configuration in one place at the start. Tomorrow: defensive programming, with preconditions, invariants, and when to use `assert`.